# Leakage Ablation -- Evaluation and Comparison

Three views of what the leakage was worth, in decreasing order of how much weight they can carry:

1. **Within-model, per class (lead with this).** One image-level model, scored on the leaked and unleaked parts of its own test set, compared class by class. Model, training, and class are all fixed; only the twin-in-training status varies.
2. **Within-model, pooled.** The same contrast without controlling for class. Bursts are unevenly spread over the classes, so this one measures composition as much as leakage -- kept only to show why the per-class version is needed.
3. **Cross-split.** Image-level against group-level results. The two splits have **different test sets**, so this mixes leakage with a difference in test composition. Indicative of magnitude, not a controlled measurement.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Pronnnnnnn/fish-freshness-mtl.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

import sys
sys.path.append('/content/repo/04_Src')

In [ ]:
DATASET_ZIP = '/content/drive/MyDrive/fish-freshness-mtl/8_fish_3_freshness.zip'
DATASET_ROOT = '/content/data/8_fish_3_freshness'
ABLATION_DIR = '/content/drive/MyDrive/fish-freshness-mtl/leakage_ablation'
CHECKPOINT_DIR = f'{ABLATION_DIR}/checkpoints'
RESULTS_DIR = f'{ABLATION_DIR}/results'
MAIN_RESULTS = '/content/drive/MyDrive/fish-freshness-mtl/results/test_results_long.csv'

import os
os.makedirs('/content/data', exist_ok=True)
!unzip -q -n "$DATASET_ZIP" -d /content/data

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score

from evaluate import load_single_task_model, predict_single_task
from metrics import classification_metrics
from comparison import to_long_format, contaminated_vs_clean, paired_difference

SPLIT_PATH = '/content/repo/08_Leakage_Ablation/02_manifests/split_manifest_imagelevel.csv'
split_df = pd.read_csv(SPLIT_PATH)
train_groups = set(split_df.loc[split_df.subset == 'train', 'time_group'])
test_df = split_df[split_df.subset == 'test'].reset_index(drop=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEEDS = [42, 43, 44]
EXPERIMENTS = [
    {'name': 'ModelA_species', 'task': 'species', 'num_classes': 8},
    {'name': 'ModelB_freshness', 'task': 'freshness', 'num_classes': 3},
]
len(test_df), DEVICE

## Evaluate the six ablation checkpoints

In [ ]:
records, predictions = [], {}

for exp in EXPERIMENTS:
    for seed in SEEDS:
        run_name = f"imglevel_{exp['name']}_seed{seed}"
        model = load_single_task_model(f'{CHECKPOINT_DIR}/{run_name}.pt', exp['num_classes'], DEVICE)
        y_true, y_pred, _ = predict_single_task(model, test_df, DATASET_ROOT, exp['task'], DEVICE)
        predictions[(exp['name'], seed)] = (exp['task'], y_true, y_pred)

        metrics = classification_metrics(y_true, y_pred, ordinal=(exp['task'] == 'freshness'))
        records.append({'model': exp['name'], 'seed': seed, 'tasks': {exp['task']: metrics}})
        print(f'[{run_name}] {exp["task"]} f1_macro = {metrics["f1_macro"]:.4f}')

imagelevel_long = to_long_format(records)
imagelevel_long.to_csv(f'{RESULTS_DIR}/imagelevel_results_long.csv', index=False)
imagelevel_long.head()

## Estimate 1 -- within-model, per class

Bursts are not spread evenly over the classes: three species never appear in the leaked portion at all, and its freshness mix leans toward Highly Fresh. Pooling across classes therefore mixes leakage with composition -- that is what makes the pooled species figure come out *negative*, which leakage cannot explain.

Comparing within each class removes it. Each row fixes the class and varies only whether the image had a twin in training. Classes absent from either portion come back as NaN and are excluded rather than averaged in.

In [ ]:
from comparison import contaminated_vs_clean_per_class

LABEL_COLUMN = {'species': 'species', 'freshness': 'freshness'}

per_class_rows = []
for (model_name, seed), (task, y_true, y_pred) in predictions.items():
    table = contaminated_vs_clean_per_class(
        test_df, train_groups, y_true, y_pred, LABEL_COLUMN[task]
    )
    table.insert(0, 'model', model_name)
    table.insert(1, 'seed', seed)
    table.insert(2, 'task', task)
    per_class_rows.append(table)

per_class_df = pd.concat(per_class_rows, ignore_index=True)
per_class_df.to_csv(f'{RESULTS_DIR}/within_model_leakage_per_class.csv', index=False)
per_class_df[per_class_df.comparable].round(4)

In [ ]:
# Macro-average the per-class inflation over classes present in both portions.
comparable = per_class_df[per_class_df.comparable]
per_run = comparable.groupby(['model', 'task', 'seed'])['inflation'].mean().reset_index()

summary = per_run.groupby(['model', 'task'])['inflation'].agg(['mean', 'std', 'min', 'max']).round(4)
summary['consistent_sign'] = per_run.groupby(['model', 'task'])['inflation'].apply(
    lambda s: bool((s > 0).all() or (s < 0).all())
)
summary['classes_used'] = comparable.groupby(['model', 'task'])['class'].nunique()
summary.to_csv(f'{RESULTS_DIR}/within_model_leakage_summary.csv')
summary

### Pooled version, kept for contrast

This is the version that ignores class composition. Its `classes_contaminated` column reads 15 against 24 for the clean portion, and the species inflation it produces is negative -- both signs that it is measuring composition, not leakage. Report the per-class figures above instead.

In [ ]:
macro_f1 = lambda a, b: f1_score(a, b, average='macro')

within_rows = []
for (model_name, seed), (task, y_true, y_pred) in predictions.items():
    result = contaminated_vs_clean(test_df, train_groups, y_true, y_pred, macro_f1)
    result.update({'model': model_name, 'seed': seed, 'task': task})
    within_rows.append(result)

within_df = pd.DataFrame(within_rows)
within_df.to_csv(f'{RESULTS_DIR}/within_model_leakage_effect.csv', index=False)
within_df[['model', 'seed', 'n_contaminated', 'n_clean', 'score_contaminated',
           'score_clean', 'inflation', 'classes_contaminated', 'classes_clean']].round(4)

In [ ]:
pooled_summary = within_df.groupby('model')['inflation'].agg(['mean', 'std', 'min', 'max']).round(4)
pooled_summary['consistent_sign'] = within_df.groupby('model')['inflation'].apply(
    lambda s: bool((s > 0).all() or (s < 0).all())
)
pooled_summary

## Estimate 2 -- cross-split: image-level vs group-level

Different test sets, so this is indicative only.

In [ ]:
main_long = pd.read_csv(MAIN_RESULTS)

rows = []
for exp in EXPERIMENTS:
    task = exp['task']
    sel = lambda df: df[(df.model == exp['name']) & (df.task == task) & (df.metric == 'f1_macro')]
    img = sel(imagelevel_long).set_index('seed')['value'].sort_index()
    grp = sel(main_long).set_index('seed')['value'].sort_index()
    diff = img - grp
    rows.append({
        'model': exp['name'], 'task': task,
        'image_level_mean': img.mean(), 'image_level_std': img.std(ddof=1),
        'group_level_mean': grp.mean(), 'group_level_std': grp.std(ddof=1),
        'mean_difference': diff.mean(), 'std_difference': diff.std(ddof=1),
        'consistent_sign': bool((diff > 0).all() or (diff < 0).all()),
        'per_seed_differences': str(diff.round(4).to_dict()),
    })

cross_df = pd.DataFrame(rows)
cross_df.round(4)

In [ ]:
CAVEAT = (
    'Cross-split rows compare models trained and evaluated on DIFFERENT test sets. '
    'The difference mixes the leakage effect with the difference in test composition '
    'and is not a controlled measurement. The within-model estimate in '
    'within_model_leakage_effect.csv holds the model and test set fixed and is the '
    'figure to lead with.'
)

cross_df['caveat'] = CAVEAT
cross_df.to_csv(f'{RESULTS_DIR}/leakage_effect_comparison.csv', index=False)
print(CAVEAT)

Saved to the ablation results folder on Drive: `imagelevel_results_long.csv`, `within_model_leakage_effect.csv`, `leakage_effect_comparison.csv`.